In [ ]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)
llm

In [2]:
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from langchain_tavily import TavilySearch

from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# -------------------------------------------------
# Initialize LLM
# -------------------------------------------------
llm = init_chat_model("groq:llama-3.1-8b-instant")

# -------------------------------------------------
# State
# -------------------------------------------------
class State(TypedDict):
    messages: Annotated[list, add_messages]

# -------------------------------------------------
# Tool 1 : Tavily Search
# -------------------------------------------------
search = TavilySearch(max_results=2)

# -------------------------------------------------
# Tool 2 : Human Assistance
# -------------------------------------------------
@tool
def human_assistance(query: str) -> str:
    """
    Ask a human for help when the AI cannot answer confidently.

    Args:
        query (str): Question to ask the human.

    Returns:
        str: Human's response.
    """
    human_response = interrupt(
        {
            "query": query
        }
    )

    return human_response["data"]

# -------------------------------------------------
# Tools
# -------------------------------------------------
tools = [search, human_assistance]

llm_with_tools = llm.bind_tools(tools)

# -------------------------------------------------
# Chatbot Node
# -------------------------------------------------
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response]
    }

# -------------------------------------------------
# Build Graph
# -------------------------------------------------
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")

builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)

builder.add_edge("tools", "chatbot")

# -------------------------------------------------
# Memory
# -------------------------------------------------
memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

# -------------------------------------------------
# Config
# -------------------------------------------------
config = {
    "configurable": {
        "thread_id": "1"
    }
}

# -------------------------------------------------
# Invoke Graph
# -------------------------------------------------
response = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="What is the recent AI news? If you don't know, ask a human."
            )
        ]
    },
    config=config,
)

print(response)

{'messages': [HumanMessage(content="What is the recent AI news? If you don't know, ask a human.", additional_kwargs={}, response_metadata={}, id='7a68128f-ce18-4c7b-a3ab-efd810b356f1'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p270hqabg', 'function': {'arguments': '{"query":"recent AI news","time_range":"week","topic":"news"}', 'name': 'tavily_search'}, 'type': 'function'}, {'id': 'eewcyjqj5', 'function': {'arguments': '{"query":"What is the recent AI news?"}', 'name': 'human_assistance'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 1812, 'total_tokens': 1863, 'completion_time': 0.097357164, 'completion_tokens_details': None, 'prompt_time': 0.126704786, 'prompt_tokens_details': None, 'queue_time': 0.162918793, 'total_time': 0.22406195}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': '

In [3]:
response = graph.invoke(
    Command(
        resume={
            "data": "OpenAI has recently released new reasoning models and announced updates to its AI platform."
        }
    ),
    config=config,
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is the recent AI news? If you don't know, ask a human.
================================== Ai Message ==================================
Tool Calls:
  tavily_search (p270hqabg)
 Call ID: p270hqabg
  Args:
    query: recent AI news
    time_range: week
    topic: news
  human_assistance (eewcyjqj5)
 Call ID: eewcyjqj5
  Args:
    query: What is the recent AI news?
================================= Tool Message =================================
Name: tavily_search

{"query": "recent AI news", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://techcrunch.com/2026/06/24/companies-are-scrambling-to-stop-employees-from-maxing-out-ai-budgets-with-small-tasks/", "title": "Companies are scrambling to stop employees from maxing out AI budgets with small tasks - TechCrunch", "score": 0.8051581, "published_date": "Wed, 24 Jun 2026 20:09:45 GMT", "content": "Founder Summit tick